# Rejected approach — can we classify the Food Category?
Our first idea was to **predict the food category** of the waste (classification). Before committing to it, we tested whether the available features can actually tell the categories apart.

In [1]:
from pathlib import Path
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

# Find the project folder (works from the repo root or from notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"
DOCS = ROOT / "docs"
MODELS = ROOT / "models"
DOCS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

TARGET = "Economic Loss (Million $)"
NUM_FEATURES = ["Year", "Total Waste (Tons)", "Avg Waste per Capita (Kg)", "Household Waste (%)"]
CAT_FEATURES = ["Country", "Food Category"]

# Load the cleaned dataset produced by 01_data_cleaning.ipynb
df = pd.read_csv(DATA / "cleaned_food_waste.csv")
print("Shape:", df.shape)
display(df.head())

Shape: (3575, 7)


,Country,Year,Food Category,Total Waste (Tons),Economic Loss (Million $),Avg Waste per Capita (Kg),Household Waste (%)
0,Australia,2021,Beverages,2967.07,2889.80,144.41,44.07
1,France,2021,Meat & Seafood,48643.30,45212.72,52.73,69.48
2,Saudi Arabia,2023,Frozen Food,1410.30,1386.31,NaN,64.73
3,UK,2023,Bakery Items,18014.05,15852.44,180.34,61.14
4,Japan,2018,Bakery Items,27534.23,29110.52,162.58,69.77


In [2]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

# Features: everything except the category itself
features_num = ["Year", "Total Waste (Tons)", "Economic Loss (Million $)",
                "Avg Waste per Capita (Kg)", "Household Waste (%)"]
features_cat = ["Country"]
X = df[features_num + features_cat]
y = df["Food Category"]

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), features_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, model in {
    "Majority-class baseline": DummyClassifier(strategy="most_frequent"),
    "Random Forest classifier": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}.items():
    scores = cross_val_score(Pipeline([("pre", pre), ("model", model)]), X, y, cv=cv, scoring="accuracy")
    results[name] = scores.mean()

results["Random guessing (1 / number of categories)"] = 1 / y.nunique()
feasibility = pd.DataFrame({"Accuracy": results}).round(3)
display(feasibility)
feasibility.to_csv(DOCS / "classification_feasibility.csv")

,Accuracy
Majority-class baseline,0.142
Random Forest classifier,0.119
Random guessing (1 / number of categories),0.125


📝 **Interpretation** how does the classifier compare to random guessing? Why did this lead us to switch to regression (predicting Economic Loss)?
1. Classifier vs random guessing
Across 8 food categories, the Random Forest model only scored 0.119 (11.9%) accuracy, falling behind both pure random guessing (0.125 / 12.5%) and simply guessing the most frequent category every time (0.142 / 14.2%). This proves the dataset's features do not hold enough information to reliably identify the food category, making classification a dead end. Instead, regression is the right choice of approach because Economic Loss is a numerical target that strongly tracks with Total Waste ($r = 0.975$), basically work in parallel .